In [ ]:
# IMPORT REQUIRED LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# CREATE / LOAD DATA

n = 200
x = np.random.uniform(1, 20, n)
# Nonlinear relationship + random noise
noise = np.random.normal(0, 15, n)
y = 20 + 5 * x + 0.8 * (x ** 2) + noise

df = pd.DataFrame({"x": x, "y": y})

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print(df.describe())

In [ ]:
# EXPLORATORY DATA ANALYSIS

plt.figure(figsize=(8, 5))

plt.scatter(df["x"], df["y"], alpha=0.6)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Relationship Between X and Y")

plt.tight_layout()
plt.show()

In [ ]:
# TRAIN / TEST SPLIT

train, test = train_test_split(df, test_size=0.20, random_state=42)

In [ ]:
print("Training observations:", len(train))
print("Testing observations :", len(test))

In [ ]:
# SIMPLE LINEAR REGRESSION

# Model:
#
#       Y = beta_0 + beta_1 X + error
#
# Ordinary Least Squares (OLS) estimates the conditional mean
# of Y given X.

linear_model = smf.ols("y ~ x", data=train).fit()

In [ ]:
linear_model.summary()

In [ ]:
# PARAMETER UNCERTAINTY

# Parameter uncertainty occurs because regression coefficients
# are estimated from a sample rather than known exactly.
#
# Standard error measures the estimated sampling variability
# of a coefficient.

print("\nEstimated coefficients:")
print(linear_model.params)

print("\nStandard errors:")
print(linear_model.bse)

print("\n95% confidence intervals:")
print(linear_model.conf_int())

In [ ]:
# CONFIDENCE INTERVALS FOR REGRESSION COEFFICIENTS

# A 95% confidence interval gives a range of plausible values
# for a population parameter under the specified model and
# inference procedure.

confidence_intervals = linear_model.conf_int(alpha=0.05)

confidence_intervals.columns = ["Lower_95%", "Upper_95%"]

print(confidence_intervals)

In [ ]:
CONFIDENCE INTERVAL FOR THE MEAN RESPONSE
# ============================================================
# The confidence interval describes uncertainty about the
# conditional MEAN response at a given value of X.
#
# It does NOT describe the range of individual future
# observations.

x_values = np.linspace(df["x"].min(), df["x"].max(), 100)

prediction_data = pd.DataFrame({"x": x_values})

prediction_results = linear_model.get_prediction(prediction_data).summary_frame(alpha=0.05)

print(prediction_results[["mean", "mean_ci_lower", "mean_ci_upper"]].head())

In [ ]:
# PREDICTION INTERVAL

# A prediction interval estimates a range in which an
# INDIVIDUAL future observation may fall.
#
# It is wider than the confidence interval for the mean
# because it includes:
#
#   1. Uncertainty in estimating the mean response
#   2. Random variation of an individual observation

prediction_results[["mean", "obs_ci_lower", "obs_ci_upper"]].head())

In [ ]:
# PLOT CONFIDENCE AND PREDICTION INTERVALS

plt.figure(figsize=(10, 6))

plt.scatter(train["x"], train["y"], alpha=0.5, label="Observed data")

plt.plot(x_values, prediction_results["mean"], label="Predicted mean")

plt.fill_between(
    x_values,
    prediction_results["mean_ci_lower"],
    prediction_results["mean_ci_upper"],
    alpha=0.3,
    label="95% Confidence Interval")

plt.fill_between(
    x_values,
    prediction_results["obs_ci_lower"],
    prediction_results["obs_ci_upper"],
    alpha=0.2,
    label="95% Prediction Interval")

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Confidence Interval vs Prediction Interval")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# MODEL UNCERTAINTY
# Model uncertainty is uncertainty about which reasonable
# model specification should be used.
#
# Here, we compare:
#
#   Model 1: Linear
#   Model 2: Quadratic
#   Model 3: Cubic
#
# If these models give substantially different predictions,
# then the choice of model structure contributes to uncertainty.

# Model 1: Linear
linear_model = smf.ols("y ~ x",data=train).fit()

# Model 2: Quadratic
quadratic_model = smf.ols("y ~ x + I(x**2)",data=train).fit()

# Model 3: Cubic
cubic_model = smf.ols("y ~ x + I(x**2) + I(x**3)",data=train).fit()

In [ ]:
# COMPARE MODEL FIT

model_comparison = pd.DataFrame({"Model": ["Linear","Quadratic","Cubic"],
    "R_squared": [linear_model.rsquared,quadratic_model.rsquared,cubic_model.rsquared],
    "Adjusted_R_squared": [linear_model.rsquared_adj,quadratic_model.rsquared_adj,cubic_model.rsquared_adj],
    "AIC": [linear_model.aic,quadratic_model.aic,cubic_model.aic],
    "BIC": [linear_model.bic,quadratic_model.bic,cubic_model.bic]})

print(model_comparison)

In [ ]:
# COMPARE MODEL PREDICTIONS

model_prediction_data = pd.DataFrame({"x": x_values})

model_predictions = pd.DataFrame({"x": x_values,
    "Linear": linear_model.predict(model_prediction_data),
    "Quadratic": quadratic_model.predict(model_prediction_data),
    "Cubic": cubic_model.predict(model_prediction_data)})

print(model_predictions.head())

In [ ]:
# VISUALIZE MODEL UNCERTAINTY

# Different fitted curves represent uncertainty about the
# functional form of the relationship.

plt.figure(figsize=(10, 6))

plt.scatter(train["x"],train["y"],alpha=0.4,label="Observed data")
plt.plot(x_values,model_predictions["Linear"],label="Linear model")
plt.plot(x_values,model_predictions["Quadratic"],label="Quadratic model")
plt.plot(x_values,model_predictions["Cubic"],label="Cubic model")

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Model Uncertainty: Alternative Model Specifications")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# QUANTIFY DIFFERENCES BETWEEN MODELS

model_predictions["Linear_vs_Quadratic"] = (model_predictions["Quadratic"] - model_predictions["Linear"])
model_predictions["Quadratic_vs_Cubic"] = (model_predictions["Cubic"] - model_predictions["Quadratic"])

print(model_predictions[["Linear_vs_Quadratic","Quadratic_vs_Cubic"]].describe())

In [ ]:
# TEST SET MODEL PERFORMANCE

linear_test_pred = linear_model.predict(test)
quadratic_test_pred = quadratic_model.predict(test)
cubic_test_pred = cubic_model.predict(test)

print("\nLinear Model")
print("RMSE:", np.sqrt(mean_squared_error(test["y"],linear_test_pred)))
print("R²:", r2_score(test["y"],linear_test_pred))

print("\nQuadratic Model")
print("RMSE:", np.sqrt(mean_squared_error(test["y"],quadratic_test_pred)))
print("R²:",r2_score(test["y"],quadratic_test_pred))

print("\nCubic Model")
print("RMSE:", np.sqrt(mean_squared_error(test["y"],cubic_test_pred)))
print("R²:", r2_score(test["y"],cubic_test_pred))

In [ ]:
# BOOTSTRAP UNCERTAINTY

# Bootstrap estimates uncertainty by repeatedly sampling
# observations WITH REPLACEMENT from the observed data.
#
# Each bootstrap sample has the same size as the original
# training dataset.

B = 5000
bootstrap_slopes = []
for i in range(B):
    # Resample rows with replacement
    bootstrap_sample = train.sample(n=len(train), replace=True)

    # Fit the regression model to the bootstrap sample
    bootstrap_model = smf.ols("y ~ x", data=bootstrap_sample).fit()

    # Store the estimated slope
    bootstrap_slopes.append(bootstrap_model.params["x"])

bootstrap_slopes = np.array(bootstrap_slopes)

In [ ]:
# BOOTSTRAP STANDARD ERROR

# The bootstrap standard error is the standard deviation of
# the bootstrap estimates.

bootstrap_se = np.std(bootstrap_slopes, ddof=1)

print("Bootstrap SE of slope:",bootstrap_se)

In [ ]:
# PERCENTILE BOOTSTRAP CONFIDENCE INTERVAL
# For a 95% percentile bootstrap interval:
#
#   Lower limit = 2.5th percentile
#   Upper limit = 97.5th percentile

bootstrap_lower = np.percentile(bootstrap_slopes,2.5)
bootstrap_upper = np.percentile(bootstrap_slopes,97.5)

print("Lower limit:",bootstrap_lower)
print("Upper limit:",bootstrap_upper)

In [ ]:
# COMPARE CLASSICAL AND BOOTSTRAP UNCERTAINTY

classical_slope = linear_model.params["x"]
classical_se = linear_model.bse["x"]
classical_lower = (classical_slope - 1.96 * classical_se)
classical_upper = (classical_slope + 1.96 * classical_se)

print("\n" + "=" * 60)
print("CLASSICAL VS BOOTSTRAP UNCERTAINTY")
print("=" * 60)

print("\nClassical slope estimate:")
print(classical_slope)

print("\nClassical standard error:")
print(classical_se)

print("\nApproximate classical 95% CI:")
print("(",classical_lower,",",classical_upper,")")

print("\nBootstrap standard error:")
print(bootstrap_se)

print("\nBootstrap 95% CI:")
print("(",bootstrap_lower,",",bootstrap_upper,")")

In [ ]:
# PLOT BOOTSTRAP DISTRIBUTION

plt.figure(figsize=(10, 6))

plt.hist(bootstrap_slopes,bins=40,alpha=0.75)
plt.axvline(classical_slope,linestyle="--",label="Original slope")
plt.axvline(bootstrap_lower,linestyle="--",label="2.5th percentile")
plt.axvline(bootstrap_upper, linestyle="--", label="97.5th percentile")

plt.xlabel("Bootstrap Regression Slope")
plt.ylabel("Frequency")
plt.title("Bootstrap Distribution of Regression Coefficient")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# BOOTSTRAP PREDICTION UNCERTAINTY

# This section estimates uncertainty in the fitted mean
# prediction caused by variation in the training sample.
#
# NOTE:
# A full individual prediction interval also includes
# residual/random outcome variation. The classical prediction
# interval calculated earlier includes that additional source.

x_new = 10
bootstrap_predictions = []

for i in range(B):
    bootstrap_sample = train.sample(n=len(train), replace=True)
    bootstrap_model = smf.ols("y ~ x", data=bootstrap_sample).fit()
    prediction = bootstrap_model.predict(pd.DataFrame({"x": [x_new]})).iloc[0]
    bootstrap_predictions.append(prediction)

bootstrap_predictions = np.array(bootstrap_predictions)
prediction_lower = np.percentile(bootstrap_predictions,2.5)
prediction_upper = np.percentile(bootstrap_predictions,97.5)

print("BOOTSTRAP PREDICTION UNCERTAINTY")
print("X value:", x_new)
print("Mean bootstrap prediction:",np.mean(bootstrap_predictions))
print("Bootstrap 95% interval:",prediction_lower,"to",prediction_upper)

In [ ]:
# QUANTILE REGRESSION

# Quantile regression models a conditional quantile of Y
# rather than only the conditional mean.
#
# Important quantiles:
#
#   tau = 0.10  -> 10th percentile
#   tau = 0.50  -> conditional median
#   tau = 0.90  -> 90th percentile

quantile_10 = smf.quantreg("y ~ x", train).fit(q=0.10)
quantile_50 = smf.quantreg("y ~ x", train).fit(q=0.50)
quantile_90 = smf.quantreg("y ~ x", train).fit(q=0.90)

In [ ]:
# DISPLAY QUANTILE REGRESSION RESULTS

print("QUANTILE REGRESSION")

print("\n10th Percentile Regression:")
print(quantile_10.summary())

print("\n50th Percentile / Median Regression:")
print(quantile_50.summary())

print("\n90th Percentile Regression:")
print(quantile_90.summary())

In [ ]:
# COMPARE QUANTILE REGRESSION COEFFICIENTS

quantile_coefficients = pd.DataFrame({"Quantile": [0.10,0.50,0.90],

    "Intercept": [
        quantile_10.params["Intercept"],
        quantile_50.params["Intercept"],
        quantile_90.params["Intercept"]],

    "X_Coefficient": [
        quantile_10.params["x"],
        quantile_50.params["x"],
        quantile_90.params["x"]]})

print("QUANTILE REGRESSION COEFFICIENTS")
print(quantile_coefficients)

In [ ]:
# PREDICT DIFFERENT CONDITIONAL QUANTILES

q10_predictions = quantile_10.predict(pd.DataFrame({"x": x_values}))
q50_predictions = quantile_50.predict(pd.DataFrame({"x": x_values}))
q90_predictions = quantile_90.predict(pd.DataFrame({"x": x_values}))

In [ ]:
PLOT QUANTILE REGRESSION
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(train["x"], train["y"], alpha=0.4, label="Observed data")
plt.plot(x_values, q10_predictions, linestyle="--", label="10th Quantile")
plt.plot(x_values, q50_predictions, label="50th Quantile / Median")
plt.plot(x_values, q90_predictions, linestyle="--", label="90th Quantile")

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Quantile Regression")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# QUANTILE REGRESSION AT MULTIPLE QUANTILES

quantiles = [0.10,0.25,0.50,0.75,0.90]
quantile_results = []

for q in quantiles:
    model = smf.quantreg("y ~ x",train).fit(q=q)

    quantile_results.append({"Quantile": q,
        "Intercept": model.params["Intercept"],
        "X_Coefficient": model.params["x"]})

quantile_results = pd.DataFrame(quantile_results)

print("MULTIPLE QUANTILE REGRESSION RESULTS")
print(quantile_results)

In [ ]:
REGRESSION DIAGNOSTICS
# ============================================================
# Residuals:
#
#       Residual = Observed value - Fitted value
#
# Residual plots help identify:
#   - Nonlinearity
#   - Heteroscedasticity
#   - Possible outliers
#   - Model misspecification

residuals = linear_model.resid
fitted_values = linear_model.fittedvalues

In [ ]:
# RESIDUALS VS FITTED VALUES

plt.figure(figsize=(8, 5))

plt.scatter(fitted_values,residuals,alpha=0.6)
plt.axhline(0,linestyle="--")

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")

plt.tight_layout()
plt.show()

In [ ]:
# 31. RESIDUAL DISTRIBUTION

plt.figure(figsize=(8, 5))

plt.hist(residuals, bins=30, alpha=0.75)

plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Distribution of Regression Residuals")

plt.tight_layout()
plt.show()